In [1]:
# Importing the global libraries
import importlib, sys, os, pandas as pd
# from dotenv import load_dotenv
import pyspark.sql as sql
from pyspark.sql import SparkSession, DataFrame, functions as F
import numpy as np
from pathlib import Path

spark = (
    SparkSession.builder
        .appName("causal_modelisation")
        .getOrCreate()
)

#Mandatory
importlib.reload(importlib)
%load_ext autoreload
%autoreload 2

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/27 13:18:30 WARN Utils: Your hostname, BEBEL-PC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/01/27 13:18:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/27 13:18:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.csv('punctuality_minimal.csv', header=True, inferSchema=True)
df.show(5)

+----------+------------+--------+------------------+------------------------+----------------------+-----------------------+---------------------+------------------+------------------------+----------------------+-----------------------+-----------------------+---------------+-------------+-----------------------+-------+
|      date|train_number|train_id|          relation|scheduled_departure_time|scheduled_arrival_time|realtime_departure_time|realtime_arrival_time|relation_direction|scheduled_departure_date|scheduled_arrival_date|realtime_departure_date|realtime_arrival_date12|delay_departure|delay_arrival|realtime_arrival_date15|station|
+----------+------------+--------+------------------+------------------------+----------------------+-----------------------+---------------------+------------------+------------------------+----------------------+-----------------------+-----------------------+---------------+-------------+-----------------------+-------+
|2026-01-16|           2|

26/01/27 13:18:39 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: date, train_number, train_id, relation, scheduled_departure_time, scheduled_arrival_time, realtime_departure_time, realtime_arrival_time, relation_direction, scheduled_departure_date, scheduled_arrival_date, realtime_departure_date, realtime_arrival_date, delay_departure, delay_arrival, realtime_arrival_date, station
 Schema: date, train_number, train_id, relation, scheduled_departure_time, scheduled_arrival_time, realtime_departure_time, realtime_arrival_time, relation_direction, scheduled_departure_date, scheduled_arrival_date, realtime_departure_date, realtime_arrival_date12, delay_departure, delay_arrival, realtime_arrival_date15, station
Expected: realtime_arrival_date12 but found: realtime_arrival_date
CSV file: file:///mnt/c/Users/bebel/OneDrive/Bureau/RailwayDT/punctuality_minimal.csv


In [3]:
df.printSchema()

root
 |-- date: date (nullable = true)
 |-- train_number: integer (nullable = true)
 |-- train_id: integer (nullable = true)
 |-- relation: string (nullable = true)
 |-- scheduled_departure_time: timestamp (nullable = true)
 |-- scheduled_arrival_time: timestamp (nullable = true)
 |-- realtime_departure_time: timestamp (nullable = true)
 |-- realtime_arrival_time: timestamp (nullable = true)
 |-- relation_direction: string (nullable = true)
 |-- scheduled_departure_date: date (nullable = true)
 |-- scheduled_arrival_date: date (nullable = true)
 |-- realtime_departure_date: date (nullable = true)
 |-- realtime_arrival_date12: date (nullable = true)
 |-- delay_departure: double (nullable = true)
 |-- delay_arrival: double (nullable = true)
 |-- realtime_arrival_date15: date (nullable = true)
 |-- station: integer (nullable = true)



In [4]:
stations_df = spark.read.csv("sumo_data/stations.csv", header=True, inferSchema=True, sep=";")
stations_df.show(5)

+----+---------+--------+-------------+-------------+------------+
|  ID|    Geo_x|   Geo_y|Symbolic_name|Name_FR_short|Name_FR_full|
+----+---------+--------+-------------+-------------+------------+
| 841|50.932897|4.216581|          LHL|       Mollem|      Mollem|
| 246|50.527312|3.526278|          FCA|   Callenelle|  Callenelle|
| 457|50.726468|4.513842|          MGV|       Genval|      Genval|
| 782|50.614049|3.800495|          FFA|       Maffle|      Maffle|
|1102|50.528178|5.219617|          LHY|       Statte|      Statte|
+----+---------+--------+-------------+-------------+------------+
only showing top 5 rows


In [5]:
stations_df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Geo_x: double (nullable = true)
 |-- Geo_y: double (nullable = true)
 |-- Symbolic_name: string (nullable = true)
 |-- Name_FR_short: string (nullable = true)
 |-- Name_FR_full: string (nullable = true)



In [6]:
# Définition d'une fenêtre partitionnée par TRAIN_NO et ordonnée par PLANNED_DATETIME_DEP

w = sql.Window.partitionBy("train_id","realtime_departure_date") \
          .orderBy("scheduled_departure_date")

df_with_next = (
    df
    .withColumn(
        "next_station",
        F.lead("station").over(w)
    )
)

df_with_next = df_with_next.alias("a").join(
    stations_df.select("id").alias("b"), 
    F.col("a.next_station") == F.col("b.id"), 
    "left"
)
# df_with_next = df_with_next.withColumnRenamed("Abbreviation_BTS_French_complete", "next_station")
df_with_next.show(20)


26/01/27 13:18:41 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: date, train_number, train_id, relation, scheduled_departure_time, scheduled_arrival_time, realtime_departure_time, realtime_arrival_time, relation_direction, scheduled_departure_date, scheduled_arrival_date, realtime_departure_date, realtime_arrival_date, delay_departure, delay_arrival, realtime_arrival_date, station
 Schema: date, train_number, train_id, relation, scheduled_departure_time, scheduled_arrival_time, realtime_departure_time, realtime_arrival_time, relation_direction, scheduled_departure_date, scheduled_arrival_date, realtime_departure_date, realtime_arrival_date12, delay_departure, delay_arrival, realtime_arrival_date15, station
Expected: realtime_arrival_date12 but found: realtime_arrival_date
CSV file: file:///mnt/c/Users/bebel/OneDrive/Bureau/RailwayDT/punctuality_minimal.csv


+----------+------------+--------+------------------+------------------------+----------------------+-----------------------+---------------------+------------------+------------------------+----------------------+-----------------------+-----------------------+---------------+-------------+-----------------------+-------+------------+----+
|      date|train_number|train_id|          relation|scheduled_departure_time|scheduled_arrival_time|realtime_departure_time|realtime_arrival_time|relation_direction|scheduled_departure_date|scheduled_arrival_date|realtime_departure_date|realtime_arrival_date12|delay_departure|delay_arrival|realtime_arrival_date15|station|next_station|  id|
+----------+------------+--------+------------------+------------------------+----------------------+-----------------------+---------------------+------------------+------------------------+----------------------+-----------------------+-----------------------+---------------+-------------+----------------------

In [7]:
# date = "2024-01-20"
chain = [911, 738, 203]

df_link_1 = (
    df_with_next
    .filter((df_with_next.station == chain[0]) & (df_with_next.next_station == chain[1]))
    # .filter(F.col("realtime_departure_date") == date)
    # .drop("train_id")
    .orderBy("scheduled_arrival_time")
)

df_link_2 = (
    df_with_next
    .filter((df_with_next.station == chain[1]) & (df_with_next.next_station == chain[2]))
    # .filter(F.col("realtime_departure_date") == date)
    # .drop("train_id")
    .orderBy("scheduled_arrival_time")
)

In [8]:
print(df_link_1.count())
df_link_1.show(5, truncate=False)
print(df_link_2.count())
df_link_2.show(5, truncate=False)

15


26/01/27 13:18:43 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: date, train_number, train_id, relation, scheduled_departure_time, scheduled_arrival_time, realtime_departure_time, realtime_arrival_time, relation_direction, scheduled_departure_date, scheduled_arrival_date, realtime_departure_date, realtime_arrival_date, delay_departure, delay_arrival, realtime_arrival_date, station
 Schema: date, train_number, train_id, relation, scheduled_departure_time, scheduled_arrival_time, realtime_departure_time, realtime_arrival_time, relation_direction, scheduled_departure_date, scheduled_arrival_date, realtime_departure_date, realtime_arrival_date12, delay_departure, delay_arrival, realtime_arrival_date15, station
Expected: realtime_arrival_date12 but found: realtime_arrival_date
CSV file: file:///mnt/c/Users/bebel/OneDrive/Bureau/RailwayDT/punctuality_minimal.csv


+----------+------------+--------+-----------------+------------------------+----------------------+-----------------------+---------------------+------------------+------------------------+----------------------+-----------------------+-----------------------+---------------+-------------+-----------------------+-------+------------+---+
|date      |train_number|train_id|relation         |scheduled_departure_time|scheduled_arrival_time|realtime_departure_time|realtime_arrival_time|relation_direction|scheduled_departure_date|scheduled_arrival_date|realtime_departure_date|realtime_arrival_date12|delay_departure|delay_arrival|realtime_arrival_date15|station|next_station|id |
+----------+------------+--------+-----------------+------------------------+----------------------+-----------------------+---------------------+------------------+------------------------+----------------------+-----------------------+-----------------------+---------------+-------------+-----------------------+---

26/01/27 13:18:45 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: date, train_number, train_id, relation, scheduled_departure_time, scheduled_arrival_time, realtime_departure_time, realtime_arrival_time, relation_direction, scheduled_departure_date, scheduled_arrival_date, realtime_departure_date, realtime_arrival_date, delay_departure, delay_arrival, realtime_arrival_date, station
 Schema: date, train_number, train_id, relation, scheduled_departure_time, scheduled_arrival_time, realtime_departure_time, realtime_arrival_time, relation_direction, scheduled_departure_date, scheduled_arrival_date, realtime_departure_date, realtime_arrival_date12, delay_departure, delay_arrival, realtime_arrival_date15, station
Expected: realtime_arrival_date12 but found: realtime_arrival_date
CSV file: file:///mnt/c/Users/bebel/OneDrive/Bureau/RailwayDT/punctuality_minimal.csv


+----------+------------+--------+-----------------+------------------------+----------------------+-----------------------+---------------------+------------------+------------------------+----------------------+-----------------------+-----------------------+---------------+-------------+-----------------------+-------+------------+---+
|date      |train_number|train_id|relation         |scheduled_departure_time|scheduled_arrival_time|realtime_departure_time|realtime_arrival_time|relation_direction|scheduled_departure_date|scheduled_arrival_date|realtime_departure_date|realtime_arrival_date12|delay_departure|delay_arrival|realtime_arrival_date15|station|next_station|id |
+----------+------------+--------+-----------------+------------------------+----------------------+-----------------------+---------------------+------------------+------------------------+----------------------+-----------------------+-----------------------+---------------+-------------+-----------------------+---

In [9]:
import plotly.express as px

# Conversion
pdf1 = df_link_1.toPandas()
pdf2 = df_link_2.toPandas()
stations = stations_df.toPandas()
stations.set_index("ID", inplace=True)

# Récupérer le link_name unique de chaque DataFrame
station_names = [stations.loc[s]["Name_FR_full"] for s in chain]

# Ajouter la colonne source avec ce link_name
pdf1["source"] = f"{station_names[0]} - {station_names[1]}"
pdf2["source"] = f"{station_names[1]} - {station_names[2]}"

# Fusionner
pdf = pd.concat([pdf1, pdf2])

# Tracer
fig = px.line(
    pdf,
    x="scheduled_arrival_time",
    y="delay_arrival",
    color="source",     # 🔑 une couleur par ligne
    # line_dash="link_name",    # 🔑 style différent selon DF
    title="Delay evolution per link"
)

fig.update_layout(
    xaxis_title="Planned datetime arrival",
    yaxis_title="Delay (seconds)",
    legend_title="Link"
)

fig.show()


26/01/27 13:18:46 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: date, train_number, train_id, relation, scheduled_departure_time, scheduled_arrival_time, realtime_departure_time, realtime_arrival_time, relation_direction, scheduled_departure_date, scheduled_arrival_date, realtime_departure_date, realtime_arrival_date, delay_departure, delay_arrival, realtime_arrival_date, station
 Schema: date, train_number, train_id, relation, scheduled_departure_time, scheduled_arrival_time, realtime_departure_time, realtime_arrival_time, relation_direction, scheduled_departure_date, scheduled_arrival_date, realtime_departure_date, realtime_arrival_date12, delay_departure, delay_arrival, realtime_arrival_date15, station
Expected: realtime_arrival_date12 but found: realtime_arrival_date
CSV file: file:///mnt/c/Users/bebel/OneDrive/Bureau/RailwayDT/punctuality_minimal.csv
26/01/27 13:18:47 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: date, trai